In [ ]:
# Środowisko — po git pull lokalnie i w Colab (ten sam URL co na slajdach)
import importlib.util
import sys
import types
import urllib.request
from pathlib import Path

KTORY = "lm"

def _mod_colab_sciezka():
    for b in [Path.cwd(), *Path.cwd().parents[:12]]:
        p = b / "notebook" / "colab_sciezka.py"
        if p.is_file():
            spec = importlib.util.spec_from_file_location("colab_sciezka", p)
            mod = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(mod)
            return mod
    mod = types.ModuleType("colab_sciezka")
    u = "https://raw.githubusercontent.com/NXTRSS/sympozjum-pediatria/main/notebook/colab_sciezka.py"
    exec(urllib.request.urlopen(u, timeout=90).read().decode(), mod.__dict__)
    sys.modules["colab_sciezka"] = mod
    return mod

_mod_colab_sciezka().bootstrap(KTORY)


In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
import numpy as np

W tym projekcie zajmiemy się technologią, która jest bardzo popularna i przydatna oraz jest jednym z głównych komponentów takich modeli jak ChatGPT — *Language Model*: [Wikipedia](https://pl.wikipedia.org/wiki/Model_językowy)<br>

W dużym uproszczeniu staramy się przewidzieć kolejne słowo w danej sekwencji. Pomyślmy o zdaniu:<br><br>
***Mamy dzisiaj bardzo słoneczną ...***<br><br>
Zapewne większość zaproponowałaby **pogodę**, ktoś **atmosferę**, a raczej nikt nie wybierze **krzesła**.<br>
Upraszczając — taką predykcję wykonują modele w stylu ChatGPT (oczywiście pozwalają też zadać pytanie — *prompt*). Nasz model ma oszacować **warunkowy rozkład** słów przy niedokończonej sekwencji:<br><br>

```
P('pogodę' | ['Mamy', 'dzisiaj', 'bardzo', 'słoneczną']) = ?

P('atmosferę' | ['Mamy', 'dzisiaj', 'bardzo', 'słoneczną']) = ?

P('krzesło' | ['Mamy', 'dzisiaj', 'bardzo', 'słoneczną']) = ?
```

Gdybyśmy zsumowali te prawdopodobieństwa po całym słowniku, otrzymalibyśmy **1**.<br><br>

Zwykle taki *Language Model* uczy się na ogromnych korpusach (np. cała Wikipedia). My „zejdzemy jedno piętro niżej”: **zamiast następnego słowa przewidujemy następną literę w wyrazie.**<br><br>
Nasz model opisuje „język” wybranego zbioru nazw z repozytorium sympozjum — i pozwoli generować nowe nazwy (domyślnie **choroby**; przełącz `ZBIOR` poniżej).

Zbiory w `data/`: `choroby.csv`, `leki.csv`, `dinozaury.csv` — w repozytorium [NXTRSS/sympozjum-pediatria](https://github.com/NXTRSS/sympozjum-pediatria). W komórce poniżej ustaw `ZBIOR = "choroby"` | `"leki"` | `"dinozaury"`.

W kolejnej komórce wczytujemy nazwy — najpierw z lokalnego pliku, w razie braku pobieramy z GitHuba.

In [2]:
# ── ZBIÓR DANYCH ──────────────────────────────────────────────
ZBIOR = "choroby"   # "leki" | "dinozaury" (pliki w data/ — lokalnie i w Colab po bootstrap)

from lm_zbiory import HIEPER, wczytaj_zbior

list_of_names, df, zrodlo, meta_zbioru = wczytaj_zbior(ZBIOR)
print(f"{meta_zbioru['emoji']} Zbiór „{ZBIOR}”: {len(list_of_names)} nazw — {meta_zbioru['opis']}")
print("Źródło:", zrodlo)
if ZBIOR != "choroby":
    print("Hiperparametry poniżej dobrane pod choroby (jak na sali); dla innego zbioru możesz dodać epoki.")
df.head()

,nazwa
0,abdominalgia
1,ablepsia
2,abortus
3,abortus habitualis
4,abrasio dentium


In [3]:
df.shape

(1292, 1)

In [4]:
list_of_names[0]   # już przetasowane (ziarno 42) w wczytaj_zbior

'ataxia teleangiectasia'

Każdą nazwę poprzedzamy tokenem **sos** (początek sekwencji) i kończymy tokenem **eos** (koniec):<br>
```
<sos>pneumonia</eos>
```
U nas: `%` na początku i `!` na końcu — wygodne przy modelu znak po znaku:<br>
```
%pneumonia!
```
Poniżej budujemy listę sekwencji treningowych.

In [5]:
name_list = ['%' + name + '!' for name in list_of_names]

name_list[:10]

['%ataxia teleangiectasia!',
 '%cephalhematoma!',
 '%dermatitis atopica!',
 '%claudicatio intermittens!',
 '%adenocarcinoma!',
 '%trachoma!',
 '%febris haemorrhagica cum nephropathia!',
 '%carcinoma anaplasticum glandulae thyroideae!',
 '%carcinoma cavi nasopharyngis!',
 '%nycturia!']

<span style="font-family:Monospace">Na sali kolejność jest zawsze taka sama (tasowanie ziarnem 42).</span>

Mamy zestaw danych — „język” nazw chorób. Teraz zbudujemy model RNN, który estymuje rozkład warunkowy kolejnych znaków.

In [6]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

import tensorflow as tf
from tensorflow.keras import Model
from tensorflow.keras.layers import Embedding, GRU, Dense
from tensorflow.keras.losses import SparseCategoricalCrossentropy
import tensorflow.keras as keras

E0000 00:00:1795766624.860741   52418 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1795766624.865134   52418 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1795766624.882122   52418 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1795766624.882142   52418 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1795766624.882144   52418 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1795766624.882145   52418 computation_placer.cc:177] computation placer already registered. Please check linka

In [7]:
import sys
from pathlib import Path

for p in (
    Path.cwd(),
    Path.cwd() / "notebook",
    Path.cwd().parent / "notebook",
    Path("/content/sympozjum-pediatria/notebook"),
):
    if (p / "lm_utils.py").exists():
        sys.path.insert(0, str(p))
        break

from lm_utils import display_generated_name, reset_seed, visualize_probabilities

SEED = 42
LICZBA_EPOK = 12
LEARNING_RATE = 0.003
TEMPERATURA = 0.4
EMBEDDING_DIM = 64
GRU_UNITS = 128
DROPOUT = 0.2

reset_seed(SEED)

# resetujemy filtr, aby nie usunął tokenów % i !
tokenizer = Tokenizer(filters='', lower=True, char_level=True)
tokenizer.fit_on_texts(name_list)

vocab_size = len(tokenizer.word_index) + 1
print(f'Wielkość słownika: {vocab_size}')

Wielkość słownika: 32


Kilka pierwszych wpisów słownika (znak → indeks):

In [8]:
Nth = ['Pierwszy','Drugi','Trzeci', 'Czwarty', 'Piąty', 'Szósty', 'Siódmy', 'Ósmy', 'Dziewiąty']
for nth, items in zip(Nth, tokenizer.word_index.items()):
    print(f'{nth:8} - {items[0]}:{items[1]}')

Pierwszy - i:1
Drugi    - a:2
Trzeci   - s:3
Czwarty  - o:4
Piąty    - e:5
Szósty   - r:6
Siódmy   - %:7
Ósmy     - !:8
Dziewiąty - t:9


Teraz przygotowujemy macierze **X** (wejście) i **Y** (cel). Dla `%pneumonia!` w jednym przebiegu sieci uczymy m.in. P(p | %), P(n | %p), … aż do P(! | …pneumonia).

```
      X      ->      Y
%pneumonia -> pneumonia!
```

RNN liczy to krok po kroku w jednym przebiegu po całej sekwencji.

**Padding** — nazwy mają różne długości. Krótsze uzupełniamy zerami na końcu; przy uczeniu zerami maskujemy (`mask_zero=True`), żeby model nie uczył się „zera → zero”.

```
%pneumonia!000000000000   →  odpowiedź przesunięta o jeden znak
```
(dopełnienie zerami tylko wyrównuje długość wierszy macierzy)

In [9]:
X_train = [name[:-1] for name in name_list]
y_train = [name[1:] for name in name_list]

przyklad = 0
print(f'Przykładowe wejście:  {X_train[przyklad]}')
print(f'Przykładowe wyjście: {y_train[przyklad]}')

X_train = tokenizer.texts_to_sequences(X_train)
y_train = tokenizer.texts_to_sequences(y_train)
X_train = pad_sequences(X_train, padding='post')
y_train = pad_sequences(y_train, padding='post')

print(f'Kształt X_train: {X_train.shape}')

Przykładowe wejście:  %ataxia teleangiectasia
Przykładowe wyjście: ataxia teleangiectasia!
Kształt X_train: (1292, 60)


In [10]:
X_train.shape

(1292, 60)

In [11]:
vocab_size

32

Konstruujemy klasę `LanguageModel` (jak na zajęciach): **Embedding** (64), **GRU** (128, dropout 0.2), **Dense** ze softmax nad słownikiem.

Ustawienia sympozjum: **12** epok, Adam **lr=0.003**, przy generowaniu temperatura **0.4**.

In [12]:
reset_seed(SEED)

class LanguageModel(Model):
    def __init__(self):
        super().__init__()
        self.embedding = Embedding(vocab_size, output_dim=EMBEDDING_DIM, mask_zero=True)
        self.rnn = GRU(GRU_UNITS, return_sequences=True, return_state=True, dropout=DROPOUT)
        self.fc = Dense(vocab_size, activation='softmax')

    def call(self, inputs, training=False, hidden_state=None, return_state=False):
        x = self.embedding(inputs, training=training)
        seq_out, last_hidden_state = self.rnn(x, training=training, initial_state=hidden_state)
        x = self.fc(seq_out)
        if return_state:
            return x, last_hidden_state
        return x

model = LanguageModel()
model(X_train)
model.summary()

Model: "language_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (1292, 60, 64)         │         2,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ ((1292, 60, 128),      │        74,496 │
│                                 │ (1292, 128))           │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (1292, 60, 32)         │         4,128 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 80,672 (315.12 KB)

 Trainable params: 80,672 (315.12 KB)

 Non-trainable params: 0 (0.00 B)

In [13]:
reset_seed(SEED)

criterion = SparseCategoricalCrossentropy()
model.compile(optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE), loss=criterion)
history = model.fit(X_train, y_train, epochs=LICZBA_EPOK, batch_size=64, verbose=2)
historia = history  # alias pod testy CI

Epoch 1/12


21/21 - 2s - 108ms/step - loss: 3.0168


Epoch 2/12


21/21 - 1s - 34ms/step - loss: 2.5248


Epoch 3/12


21/21 - 1s - 33ms/step - loss: 2.3453


... (14 outputs left out)

Epoch 11/12


21/21 - 1s - 33ms/step - loss: 1.8271


Epoch 12/12


21/21 - 1s - 34ms/step - loss: 1.7728


##### Jak zmieniał się błąd uczenia w kolejnych epokach

In [14]:
loss = history.history['loss']

plt.figure(figsize=(6, 4))
plt.plot(loss, label='Błąd uczenia', color='blue')
plt.xlabel('Epoki')
plt.ylabel('Błąd')
plt.title('Błąd uczenia w trakcie epok')
plt.legend()
plt.grid(True)
plt.show()

/tmp/ipykernel_52418/244929200.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Po treningu model jest **generatywny** — tworzy nowe nazwy znak po znaku.

Na każdym kroku podajemy ostatni wygenerowany znak i ukryty stan GRU z poprzedniego kroku. Losowanie z rozkładu regulujemy **temperaturą** (u nas 0.4). Przy `visuals=True` na końcu jedna plansza z histogramami wszystkich kroków (`histogramy_krokow` w `lm_wizualizacja.py`).


In [15]:
def sample_model_optimal(
    prompt='%',
    greedy=False,
    temperature=TEMPERATURA,
    print_outcome=True,
    visuals=False,
    known_names=None,
):
    from lm_wizualizacja import histogramy_krokow

    prompt = prompt.lower()
    id2znak = tokenizer.index_word
    kroki = [] if visuals else None

    def _zapisz_krok(prefiks, rozklad, wybor):
        pelny = np.zeros(vocab_size, dtype=np.float64)
        r = np.asarray(rozklad, dtype=np.float64).ravel()
        pelny[: min(len(r), vocab_size)] = r[:vocab_size]
        kroki.append({"kontekst": prefiks, "rozklad": pelny, "wybor": int(wybor)})

    encoded = tokenizer.texts_to_sequences([prompt])
    encoded = tf.convert_to_tensor(encoded)
    model_out, hidden_state = model(encoded, return_state=True)
    last_output = model_out[0, -1, :].numpy()

    for i in range(30):
        last_output = np.array(last_output, dtype=np.float64)
        last_output[0] = 0.0
        if last_output.sum() > 0:
            last_output = last_output / last_output.sum()

        if greedy:
            next_char_idx = int(last_output.argmax())
            rozklad_wiz = last_output.copy()
        else:
            log_p = np.log(last_output + 1e-9) / max(temperature, 1e-6)
            rozklad_wiz = np.exp(log_p)
            rozklad_wiz[0] = 0.0
            rozklad_wiz = rozklad_wiz / rozklad_wiz.sum()
            next_char_idx = int(np.random.choice(np.arange(len(rozklad_wiz)), p=rozklad_wiz))

        if visuals:
            _zapisz_krok(prompt, rozklad_wiz, next_char_idx)

        next_char = tokenizer.index_word[next_char_idx]
        prompt += next_char

        if next_char == '!':
            break

        encoded = tokenizer.texts_to_sequences([next_char])
        encoded = tf.convert_to_tensor(encoded)
        model_out, hidden_state = model(encoded, hidden_state=hidden_state, return_state=True)
        last_output = model_out[0, -1, :].numpy()

    if print_outcome:
        nazwa = prompt[1:-1]
        display(Markdown(f"## {meta_zbioru['emoji']} **Wygenerowana nazwa {meta_zbioru['gen_etykieta']}:** `{nazwa}`"))
        display_generated_name(prompt, known_names=known_names)
    if visuals and kroki:
        histogramy_krokow(kroki, id2znak, vocab_size)
    return prompt


In [16]:
reset_seed(SEED)
prompt = '%'
generated_text = sample_model_optimal(
    prompt, visuals=True, temperature=TEMPERATURA, known_names=set(list_of_names)
)
pierwsza_nazwa = generated_text[1:-1]

## 🩺 **Wygenerowana nazwa choroby:** `syndroma lagus`

✨ „syndroma lagus” — nie ma w zbiorze (model coś wymyślił)


/workspace/notebook/lm_wizualizacja.py:69: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Dziesięć nazw** — szybka kontrola na sali: jeśli nazwa była w zbiorze treningowym, zobaczysz ⚠️; jeśli model coś wymyślił — ✨.

In [17]:
reset_seed(SEED)
lista_nazw = []
znane = set(list_of_names)
for _ in range(10):
    wynik = sample_model_optimal(
        '%',
        print_outcome=False,
        visuals=False,
        temperature=TEMPERATURA,
        known_names=znane,
    )
    lista_nazw.append(wynik[1:-1])
    display_generated_name(wynik, known_names=znane)

✨ „syndroma lagus” — nie ma w zbiorze (model coś wymyślił)


✨ „angiomatosis” — nie ma w zbiorze (model coś wymyślił)
✨ „carcinoma pertica” — nie ma w zbiorze (model coś wymyślił)


✨ „syndroma dissenta” — nie ma w zbiorze (model coś wymyślił)
✨ „syndroma pronica” — nie ma w zbiorze (model coś wymyślił)


✨ „carcinoma pronica” — nie ma w zbiorze (model coś wymyślił)
✨ „asterosis” — nie ma w zbiorze (model coś wymyślił)
✨ „anthratio certica” — nie ma w zbiorze (model coś wymyślił)


✨ „parcinoma pollinatio atri” — nie ma w zbiorze (model coś wymyślił)
✨ „tystis acuta” — nie ma w zbiorze (model coś wymyślił)
